Refactored de-noising script.

[Runtime: ~1.5min per scan file]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, sys, csv, json, math, glob, re, time, warnings
from pathlib import Path
import subprocess
import shlex
from collections import Counter

try:
    cores = os.cpu_count()
except Exception:
    phys = 2
os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = str(cores)
os.environ["OMP_NUM_THREADS"] = str(cores)
os.environ["MKL_NUM_THREADS"] = str(cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(cores)
print(f"[Parallelization:] Planning to use {cores} processing threads.")

import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to
from nibabel.orientations import io_orientation, axcodes2ornt, inv_ornt_aff, apply_orientation, aff2axcodes

from scipy import ndimage as ndi
from scipy.ndimage import gaussian_gradient_magnitude, binary_dilation, binary_erosion, binary_opening, binary_closing, binary_fill_holes
import matplotlib.pyplot as plt
from nilearn import plotting

from sklearn.decomposition import PCA
from nilearn.signal import clean

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
### General parameters:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

OVERWRITE = config['overwrite_BBR']

### Processing parameters:

#___________________________________
# Coregistration (BBR) parameters:
WM_LABELS  = config['BBR_parameters']['WM_labels']
CSF_LABELS = config['BBR_parameters']['CSF_labels']
ERODE_WM_VOX  = config['BBR_parameters']['erode_WM_vox']     # was '1'; at '4' mm this was too aggressive
ERODE_CSF_VOX = config['BBR_parameters']['erode_CSF_vox']
CLIP_TO_EPI_SUPPORT = config['BBR_parameters']['clip_to_EPI_support_mask']
EPI_SUPPORT_PCT = config['BBR_parameters']['EPI_support_threshold']

#___________________________________
### De-noising parameters:
SCRUB_MODE       = config['denoising_parameters']['scrub_mode']
NEIGHBOR_EXPAND  = config['denoising_parameters']['neighbor_expand']
APPLY_GSR        = config['denoising_parameters']['apply_GSR']
GSR_TISSUE       = config['denoising_parameters']['GSR_tissue']
# aCompCor components:
ACOMPCOR_N_WM    = config['denoising_parameters']['aCompCor_WM_components']
ACOMPCOR_N_CSF   = config['denoising_parameters']['aCompCor_CSF_components']
# Band-pass (Hz):
HP_HZ            = config['denoising_parameters']['denoise_HPF']
LP_HZ            = config['denoising_parameters']['denoise_LPF']
# Detrend inside nilearn.clean:
DETREND          = config['denoising_parameters']['detrend']
# Censoring-related thresholds (???):
FD_RELAXED_PERC  = config['denoising_parameters']['FD_perc_censor_threshold']
DVARS_RELAXED_P  = config['denoising_parameters']['DVARS_perc_censor_threshold']
# Optional absolute thresholds (None = off):
FD_ABS_MM        = config['denoising_parameters']['FD_abs_censor_threshold']
if FD_ABS_MM in ['None', 'none', ""]:
    FD_ABS_MM = None
DVARS_ABS        = config['denoising_parameters']['DVARS_abs_censor_threshold']
if DVARS_ABS in ['None', 'none', ""]:
    DVARS_ABS = None


# __________________________________________________________________________________________________________
### Set filepaths:

### INPUTS:
BASE_DIRECTORY = config['root_output_directory']

RUN_MANIFEST_PATH = Path(BASE_DIRECTORY) / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = Path(BASE_DIRECTORY) / 'fMRI_manifest.csv'

SDC_DIRECTORY = Path(BASE_DIRECTORY) / config['SDC_output_dir']

### OUTPUTS:
BBR_OUTPUT_DIR = Path(BASE_DIRECTORY) / config['BBR_output_dir']
BBR_QC_DIR = Path(BASE_DIRECTORY) / config['BBR_QC_subdir']

DENOISING_OUTPUT_DIR = Path(BASE_DIRECTORY) / config['denoising_output_dir']
DENOISING_QC_DIR = Path(BASE_DIRECTORY) / config['denoising_QC_subdir']

### INITIALIZE:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

-------

Initial data audit / check:

In [ ]:
# __________________________________________________________________________________________________________
### Ensure output directories exist:

for d in [BBR_OUTPUT_DIR, BBR_QC_DIR, DENOISING_OUTPUT_DIR, DENOISING_QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# __________________________________________________________________________________________________________
### Attach required filepaths to fMRI_runs and perform precheck:

def _exists(p):
    try:
        return os.path.exists(p)
    except Exception:
        return False

# Construct 'run_prefix' based on format '<subject_ID>_<session_ID>_*':
fMRI_runs["run_prefix"] = fMRI_runs.apply(
    lambda r: f"{r['subject_ID']}_{r['session_ID']}",
    axis=1)

# Set SDC output directory and boldref image:
fMRI_runs["sdc_run_dir"] = fMRI_runs["run_prefix"].apply(
    lambda p: SDC_DIRECTORY / p)
fMRI_runs["boldref_sdc"] = fMRI_runs["sdc_run_dir"].apply(
    lambda d: d / "boldref_sdc.nii")

# Set FreeSurfer subject directory + key MGZ files:
fs_subj_root = Path(FREESURFER_SUBJECTS_DIR)
fMRI_runs["fs_subj_dir"]  = fMRI_runs["subject_ID"].apply(
    lambda sid: fs_subj_root / sid)
fMRI_runs["fs_brain_mgz"] = fMRI_runs["fs_subj_dir"].apply(
    lambda d: d / "mri" / "brain.mgz")
fMRI_runs["fs_brainmask"] = fMRI_runs["fs_subj_dir"].apply(
    lambda d: d / "mri" / "brainmask.mgz")
fMRI_runs["fs_aseg_mgz"]  = fMRI_runs["fs_subj_dir"].apply(
    lambda d: d / "mri" / "aseg.mgz")

# File-existence flags:
check_cols = ["boldref_sdc", "fs_subj_dir", "fs_brain_mgz", "fs_brainmask", "fs_aseg_mgz"]
for c in check_cols:
    fMRI_runs[f"has_{c}"] = fMRI_runs[c].apply(_exists)

# Enforce critical inputs:
#   - boldref_sdc (moving image for bbregister)
#   - fs_subj_dir (to find FS surfaces/registration context)
#   - fs_brain_mgz (sanity-check FS anatomy is present)
critical = ["boldref_sdc", "fs_subj_dir", "fs_brain_mgz"]

missing_rows = []
for _, r in fMRI_runs.iterrows():
    miss = [c for c in critical if not r[f"has_{c}"]]
    if miss:
        missing_rows.append({
            "subject_ID": r["subject_ID"],
            "group_ID":   r.get("group_ID", None),
            "session_ID": r["session_ID"],
            "missing":    miss})
problems_df = None
if missing_rows:
    problems_df = pd.DataFrame(missing_rows)

    lines = ["[BBR PRECHECK] Missing critical inputs for the following entries:"]
    for m in missing_rows:
        tag = f"{m['group_ID']}_{m['subject_ID']}_{m['session_ID']}"
        lines.append(f"  - {tag}: " + ", ".join(m["missing"]))

    if HARD_STOP:
        lines.append("HARD_STOP is True; fix paths or re-run prior steps, then re-execute this cell.")
        # Print the detailed list for visibility before raising:
        print("\n".join(lines))
        raise RuntimeError("[BBR PRECHECK] Critical inputs missing; see problems_df for details.")
    else:
        lines.append("HARD_STOP is False; these runs will be dropped and processing will continue.")
        warnings.warn("\n".join(lines))

        # Drop failing (subject_ID, session_ID) pairs from 'fMRI_runs':
        bad_keys = {(m["subject_ID"], m["session_ID"]) for m in missing_rows}
        mask = ~fMRI_runs.apply(
            lambda r: (r["subject_ID"], r["session_ID"]) in bad_keys,
            axis=1)
        n_dropped = (~mask).sum()
        fMRI_runs = fMRI_runs[mask].reset_index(drop=True)
        print(f"[BBR PRECHECK] Dropped {n_dropped} runs due to missing critical inputs; "
              f"{len(fMRI_runs)} runs remain.")

else:
    print("[BBR PRECHECK] All critical inputs found for all runs in fMRI_runs.")

# Sort fMRI_runs for downstream processing & preview (replaces old df_ready)
fMRI_runs = fMRI_runs.sort_values(by=["group_ID", "subject_ID"]).reset_index(drop=True)

print(f"[BBR PRECHECK] Validated data prerequisites for {len(fMRI_runs)} subjects \ files.")

preview_cols = [
    "subject_ID", "group_ID", "session_ID",
    "boldref_sdc", "fs_subj_dir", "fs_brain_mgz", "fs_brainmask", "fs_aseg_mgz"]
# display(fMRI_runs[preview_cols])

In [ ]:
# === STEP BBR-1: Run bbregister to estimate EPI→T1 rigid (reg.dat) per run ===
# Uses the SDC-corrected boldref as the moving image and FreeSurfer's subject surfaces/T1 as fixed.
# Outputs:
#   BBR_OUTPUT_DIR/<subject>_<session>/reg.dat
#   BBR_OUTPUT_DIR/<subject>_<session>/bbregister.log

def _ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def _bbregister_available() -> str:
    # Return path to bbregister (if available under current FS env); raise hard error if not found:
    cmd = "which bbregister"
    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError(f"Cannot find bbregister under current environment.\nSTDERR:\n{r.stderr}")
    return r.stdout.strip()

def _find_boldref_sdc(run_dir: str):
    # Find the corresponding SDC-corrected boldref for each analysis target:
    if not run_dir:
        return None
    for name in ("boldref_sdc.nii", "boldref_sdc.nii.gz", "boldref_sdc.mgz"):
        p = os.path.join(run_dir, name)
        if os.path.exists(p):
            return p
    return None

def run_bbregister_for_row(row, overwrite=True):
    subj   = str(row["subject_ID"])
    group  = str(row["group_ID"])
    sess   = str(row["session_ID"])
    mov    = str(row["boldref_sdc"])     # <-- moving (EPI, SDC-corrected)
    fs_dir = str(row["fs_subj_dir"])     # <-- SUBJECTS_DIR/<subject_ID>

    # New run_tag naming: '{<subject>_<session>}' format
    run_tag = f"{subj}_{sess}"
    out_dir = BBR_OUTPUT_DIR / run_tag
    _ensure_dir(out_dir)

    out_reg = out_dir / "reg.dat"
    out_log = out_dir / "bbregister.log"

    if (not overwrite) and out_reg.exists():
        print(f"[BBR] {run_tag}: reg.dat exists — skipping (OVERWRITE=False).")
        return {
            "subject_ID": subj, "group_ID": group, "session_ID": sess,
            "sdc_run_dir": str(row["sdc_run_dir"]), "fs_subj_dir": fs_dir, "boldref_sdc": mov,
            "run_dir": str(out_dir), "reg_dat": str(out_reg), "status": "exists"}

    # Build bbregister command:
    # '--bold' selects EPI contrast; '--init-coreg' is a robust starting point for BOLD→T1
    bbreg_cmd = (
        f'bbregister --s {shlex.quote(subj)} '
        f'--mov {shlex.quote(mov)} '
        f'--bold --init-coreg '
        f'--reg {shlex.quote(str(out_reg))}')

    # Execute under current FS environment:
    wrapped = (
        f'echo "[CMD] {bbreg_cmd}" > {shlex.quote(str(out_log))} 2>&1; '
        f'{bbreg_cmd} >> {shlex.quote(str(out_log))} 2>&1')
    print(f"[BBR] {run_tag}: bbregister ...")
    r = subprocess.run(["bash", "-lc", wrapped], text=True)

    if r.returncode != 0 or (not out_reg.exists()):
        tail = ""
        try:
            with open(out_log, "r") as f:
                lines = f.readlines()[-40:]
            tail = "".join(lines)
        except Exception:
            pass
        raise RuntimeError(
            f"[BBR FAIL] {run_tag}: bbregister did not produce reg.dat.\n"
            f"Log tail:\n{tail}")

    print(f"[BBR OK] {run_tag}: wrote {out_reg}")
    return {
        "subject_ID": subj, "group_ID": group, "session_ID": sess,
        "sdc_run_dir": str(row["sdc_run_dir"]), "fs_subj_dir": fs_dir, "boldref_sdc": mov,
        "run_dir": str(out_dir), "reg_dat": str(out_reg), "status": "ok"}

bbreg_path = _bbregister_available()
print(f"[BBR] Using: {bbreg_path}")

# Ensure fMRI_runs is present and non-empty (precheck cell should have run already):
if "fMRI_runs" not in globals() or fMRI_runs.empty:
    raise RuntimeError("fMRI_runs is missing/empty — run the precheck cell first.")

# Ensure required columns exist (note: uses sdc_run_dir, not sdc_dir):
required_cols = ["subject_ID", "group_ID", "session_ID", "sdc_run_dir", "fs_subj_dir"]
missing = [c for c in required_cols if c not in fMRI_runs.columns]
if missing:
    raise KeyError(f"fMRI_runs missing required columns: {missing}. Present: {list(fMRI_runs.columns)}")

# Derive/ensure 'boldref_sdc' exists (use existing if present):
if "boldref_sdc" not in fMRI_runs.columns:
    fMRI_runs["boldref_sdc"] = fMRI_runs["sdc_run_dir"].apply(_find_boldref_sdc)

# Validate critical inputs per row (throws hard error if anything is missing):
problems = []
for _, r in fMRI_runs.iterrows():
    tag = f"{r['group_ID']}_{r['subject_ID']}_{r['session_ID']}"
    # boldref_sdc
    if not r.get("boldref_sdc") or not os.path.exists(r["boldref_sdc"]):
        problems.append(f"{tag}: boldref_sdc missing under {r['sdc_run_dir']}")
    # FS anatomy (bbregister expects SUBJECTS_DIR/<subject_ID>/mri/brain*.mgz)
    fs_brain = os.path.join(r["fs_subj_dir"], "mri", "brain.mgz")
    fs_mask  = os.path.join(r["fs_subj_dir"], "mri", "brainmask.mgz")
    if not os.path.exists(fs_brain):
        problems.append(f"{tag}: FreeSurfer brain.mgz missing ({fs_brain})")
    if not os.path.exists(fs_mask):
        problems.append(f"{tag}: FreeSurfer brainmask.mgz missing ({fs_mask})")
if problems:
    raise FileNotFoundError("Critical inputs missing for bbregister:\n  - " + "\n  - ".join(problems))

# Run bbregister for all (already subsetted) runs in fMRI_runs:
bbr_results = []
for _, row in fMRI_runs.iterrows():
    res = run_bbregister_for_row(row, overwrite=OVERWRITE)
    bbr_results.append(res)

# Build final dataframe with the fields QC will need; sort by subject_ID:
bbr_df = (pd.DataFrame(bbr_results).sort_values("subject_ID").reset_index(drop=True))
# Tidy view:
view_cols = ["subject_ID", "group_ID", "session_ID", "boldref_sdc", "reg_dat", "fs_subj_dir", "sdc_run_dir", "status"]
bbr_df[view_cols]

In [ ]:
# === STEP BBR-2 (QC-forward, RAS+ clean view): EPI→T1 overlay with better orientation/contrast ===
# Outputs:
#   BBR_OUTPUT_DIR/<subject>_<session>/epi_in_t1.nii.gz
#   BBR_QC_DIR/<subject>_<session>_06_bbr_overlay_forward.png

def _ensure_dir(p): Path(p).mkdir(parents=True, exist_ok=True)

def _run(cmd):
    r = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed:\n{cmd}\nSTDOUT:\n{r.stdout}\nSTDERR:\n{r.stderr}")
    return r

def _robust_range(x, mask=None, p_lo=2.0, p_hi=98.0):
    if mask is not None:
        x = x[mask]
    x = x[np.isfinite(x)]
    if x.size == 0:
        return 0.0, 1.0
    lo, hi = np.percentile(x, p_lo), np.percentile(x, p_hi)
    if not np.isfinite(hi - lo) or (hi - lo) < 1e-6:
        lo, hi = float(np.nanmin(x)), float(np.nanmax(x))
        if not np.isfinite(hi - lo) or (hi - lo) < 1e-6:
            hi = lo + 1.0
    return float(lo), float(hi)

def _to_ras(img):
    # Reorient to RAS+ for DISPLAY ONLY (does not touch files on disk):
    return nib.as_closest_canonical(img)

def _overlay_three_views_filled(epi_img, t1_img, mask_img, out_png, title):
    # Canonicalize to RAS+ for consistent viewing:
    epi_c = _to_ras(epi_img)
    t1_c  = _to_ras(t1_img)
    msk_c = _to_ras(mask_img)

    epi = epi_c.get_fdata()
    t1  = t1_c.get_fdata()
    msk = (msk_c.get_fdata() > 0)

    # Choose slice indices around the mask bbox center (or volume center):
    nz = np.array(np.where(msk))
    if nz.size > 0:
        cx = int((nz[0].min() + nz[0].max()) / 2)
        cy = int((nz[1].min() + nz[1].max()) / 2)
        cz = int((nz[2].min() + nz[2].max()) / 2)
    else:
        sx, sy, sz = epi.shape
        cx, cy, cz = sx // 2, sy // 2, sz // 2

    # Robust windowing:
    lo_t1, hi_t1 = _robust_range(t1, mask=msk, p_lo=1.0, p_hi=99.0)
    lo_e,  hi_e  = _robust_range(epi, mask=msk, p_lo=2.0, p_hi=98.0)

    # Prepare panels in RAS+ coordinates:
    # sagittal = X-constant (left-right), coronal = Y-constant (posterior-anterior), axial = Z-constant (inferior-superior)
    sag_t1, sag_epi = t1[cx, :, :].T, epi[cx, :, :].T
    cor_t1, cor_epi = t1[:, cy, :].T, epi[:, cy, :].T
    axi_t1, axi_epi = t1[:, :, cz],   epi[:, :, cz]

    # Mask for display scaling only (don’t hide EPI outside brain in overlay):
    epi_alpha = 0.45

    fig = plt.figure(figsize=(12, 4), constrained_layout=True)
    gs  = fig.add_gridspec(1, 3)
    axs = [fig.add_subplot(gs[0, i]) for i in range(3)]
    panels = [("Sagittal", sag_t1, sag_epi),
              ("Coronal",  cor_t1, cor_epi),
              ("Axial",    axi_t1, axi_epi)]

    im = None
    for ax, (lab, bg, fg) in zip(axs, panels):
        ax.imshow(bg, cmap="gray", vmin=lo_t1, vmax=hi_t1, origin="lower", interpolation="nearest")
        im = ax.imshow(fg, cmap="magma", vmin=lo_e,  vmax=hi_e,  origin="lower", alpha=epi_alpha, interpolation="nearest")
        ax.set_title(lab)
        ax.axis("off")

    cbar = fig.colorbar(im, ax=axs, fraction=0.025, pad=0.02)
    cbar.set_label("EPI intensity", rotation=270, labelpad=12)
    fig.suptitle(title, fontsize=12)
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

# Main execution -- warp EPI→T1 and generate overlays:
if 'bbr_df' not in globals() or bbr_df.empty:
    raise RuntimeError("bbr_df missing/empty — run the BBR step first.")

for r in bbr_df.to_dict(orient="records"):
    subj, grp, sess = r["subject_ID"], r["group_ID"], r["session_ID"]

    run_tag   = f"{subj}_{sess}"

    epi_path  = r["boldref_sdc"]
    reg_dat   = r["reg_dat"]
    fs_dir    = r["fs_subj_dir"]
    brain_mgz = os.path.join(fs_dir, "mri", "brain.mgz")
    mask_mgz  = os.path.join(fs_dir, "mri", "brainmask.mgz")

    run_dir   = Path(r.get("run_dir") or (BBR_OUTPUT_DIR / run_tag))
    _ensure_dir(run_dir)
    epi_in_t1 = run_dir / "epi_in_t1.nii.gz"
    png       = BBR_QC_DIR / f"{run_tag}_06_bbr_overlay_forward.png"

    # EPI (mov) -> T1 (targ): forward mapping:
    cmd = (
        f'mri_vol2vol --mov {shlex.quote(epi_path)} '
        f'--targ {shlex.quote(brain_mgz)} '
        f'--reg {shlex.quote(reg_dat)} '
        f'--o {shlex.quote(str(epi_in_t1))} --interp trilinear')
    _run(cmd)

    # Load and show orientation codes (to sanity-check flips):
    epi_t1_img = nib.load(str(epi_in_t1))
    t1_img     = nib.load(brain_mgz)
    msk_img    = nib.load(mask_mgz)
    print(f"[ORIENT] {run_tag}: EPI_in_T1 {aff2axcodes(epi_t1_img.affine)} | T1 {aff2axcodes(t1_img.affine)}")

    # Generate overlay figure:
    _overlay_three_views_filled(epi_t1_img, t1_img, msk_img,
                                str(png), title=f"{run_tag} — BBR QC (forward)")
    print(f"[BBR QC] {run_tag}: wrote {png}")

Assuming the overlays from the above QC look good, next step is to actually write new anatomical brain masks (incl. separate brain-, white matter- and CSF/ventricle masks) that are based in EPI space.

(In the previous steps we were simply using BBRegister to pre-compute these transformations, whereas now we're using these to write new brain-volume objects directly into EPI space.)

We also run some quick QC sanity-checks to verify appropriate alignment & coverage before moving on to de-noising.

In [ ]:
# === STEP BBR-3 (final tidy): FS masks T1→EPI via reg.dat→LTA→ITK, apply on EPI grid
# Tweaks:
#   - Disabled erosions at 4 mm (ERODE_* = 0).
#   - Optional: clip the projected brainmask to an EPI support mask (derived from boldref)

def _run(cmd: str):
    r = subprocess.run(["bash","-lc", cmd], text=True, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed:\n{cmd}\nSTDOUT:\n{r.stdout}\nSTDERR:\n{r.stderr}")
    return r

def _which(bin_name: str) -> str:
    r = subprocess.run(["bash","-lc", f"command -v {shlex.quote(bin_name)} || true"],
                       text=True, capture_output=True)
    return r.stdout.strip()

def _find_ants_apply_or_none():
    p = _which("antsApplyTransforms")
    if p: 
        return p
    if os.environ.get("ANTSPATH"):
        c = os.path.join(os.environ["ANTSPATH"], "antsApplyTransforms")
        if os.path.exists(c): 
            return c
    for c in ["/opt/ants/bin/antsApplyTransforms","/opt/ANTs/bin/antsApplyTransforms",
              "/usr/local/ants/bin/antsApplyTransforms","/usr/local/ANTs/bin/antsApplyTransforms",
              "/usr/local/bin/antsApplyTransforms","/usr/bin/antsApplyTransforms"]:
        if os.path.exists(c): 
            return c
    return None

ANTS_APPLY = _find_ants_apply_or_none()

def _mk_fs_binary(fs_subj_dir, out_dir, labels, name):
    aseg = os.path.join(fs_subj_dir, "mri", "aseg.mgz")
    out  = os.path.join(out_dir, f"{name}_mask_fs.mgz")
    if not os.path.exists(out):
        cmd = (
            f'mri_binarize --i {shlex.quote(aseg)} --match ' +
            " ".join(map(str, labels)) +
            f' --o {shlex.quote(out)}')
        _run(cmd)
    return out

def _regdat_to_itk_t1_to_epi(regdat, mov_epi, targ_t1, out_itk_txt, tmp_dir):
    lta_e2t = os.path.join(tmp_dir, "epi2t1.lta")
    lta_t2e = os.path.join(tmp_dir, "t1toepi.lta")

    cmd1 = (
        f'tkregister2 --noedit --reg {shlex.quote(regdat)} '
        f'--mov {shlex.quote(mov_epi)} --targ {shlex.quote(targ_t1)} '
        f'--ltaout {shlex.quote(lta_e2t)}')
    _run(cmd1)

    cmd2 = (
        f'lta_convert --inlta {shlex.quote(lta_e2t)} --invert '
        f'--outlta {shlex.quote(lta_t2e)}')
    _run(cmd2)

    cmd3 = (
        f'lta_convert --inlta {shlex.quote(lta_t2e)} '
        f'--src {shlex.quote(targ_t1)} --trg {shlex.quote(mov_epi)} '
        f'--outitk {shlex.quote(out_itk_txt)}')
    _run(cmd3)

    if not os.path.exists(out_itk_txt):
        raise FileNotFoundError(f"ITK transform not written: {out_itk_txt}")

def _fs_mgz_to_nii(src_mgz, out_nii):
    if not os.path.exists(out_nii):
        cmd = f'mri_convert {shlex.quote(src_mgz)} {shlex.quote(out_nii)}'
        _run(cmd)

def _ants_apply_nn(img_t1_nii, ref_epi_nii, itk_t1_to_epi, out_nii):
    if ANTS_APPLY:
        cmd = (
            f'{shlex.quote(ANTS_APPLY)} -d 3 '
            f'-i {shlex.quote(img_t1_nii)} '
            f'-r {shlex.quote(ref_epi_nii)} '
            f'-t {shlex.quote(itk_t1_to_epi)} '
            f'-o {shlex.quote(out_nii)} '
            f'-n NearestNeighbor')
        _run(cmd)
    else:
        import ants
        fixed  = ants.image_read(ref_epi_nii)
        moving = ants.image_read(img_t1_nii)
        out    = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=[itk_t1_to_epi],
            interpolator="nearestNeighbor")
        ants.image_write(out, out_nii)

def _fractions(epi_img, brain_img, wm_img, csf_img):
    # [Vestigal function]
    b = brain_img.get_fdata() > 0.5
    tot = int(b.sum())
    if tot == 0: 
        return 0.0, 0.0, 0.0
    w = (wm_img.get_fdata()  > 0.5) & b
    c = (csf_img.get_fdata() > 0.5) & b
    g = b & (~w) & (~c)
    return 100 * w.sum() / tot, 100 * g.sum() / tot, 100 * c.sum() / tot

def _maybe_erode(img, iters):
    if iters <= 0: 
        return img
    arr = img.get_fdata() > 0.5
    arr = binary_erosion(arr, iterations=int(iters))
    return nib.Nifti1Image(arr.astype(np.uint8), img.affine, img.header)

def _epi_support_mask(epi_img):
    # Very conservative head/brain support from EPI intensities to clip FS brainmask:
    e = epi_img.get_fdata()
    v = e[np.isfinite(e)]
    if v.size == 0:
        m = np.zeros(epi_img.shape, dtype=bool)
    else:
        thr = np.percentile(v, 35)  # conservative
        m = e > thr
        m = binary_opening(m, iterations=1)
        m = binary_closing(m, iterations=2)
        m = binary_fill_holes(m)
        m = binary_dilation(m, iterations=1)
    return nib.Nifti1Image(m.astype(np.uint8), epi_img.affine, epi_img.header)


# ______________________________________________________________________________________________________
# Main execution:
if 'bbr_df' not in globals() or bbr_df.empty:
    raise RuntimeError("bbr_df missing/empty — run the BBR step first.")

rows = bbr_df.to_dict(orient="records")
summ = []

for r in rows:
    subj, grp, sess = str(r["subject_ID"]), str(r["group_ID"]), str(r["session_ID"])
    tag = f"{subj}_{sess}"

    epi    = str(r["boldref_sdc"])
    fs_dir = str(r["fs_subj_dir"])

    run_dir = BBR_OUTPUT_DIR / tag
    run_dir = str(run_dir)
    Path(run_dir).mkdir(parents=True, exist_ok=True)

    regdat = os.path.join(run_dir, "reg.dat")
    if not os.path.exists(regdat):
        raise FileNotFoundError(f"{tag}: missing reg.dat at {regdat}")

    t1_brain = os.path.join(fs_dir, "mri", "brain.mgz")
    bm_mgz   = os.path.join(fs_dir, "mri", "brainmask.mgz")
    wm_mgz   = _mk_fs_binary(fs_dir, run_dir, WM_LABELS,  "wm")
    csf_mgz  = _mk_fs_binary(fs_dir, run_dir, CSF_LABELS, "csf")

    # Convert T1 masks to NIfTI:
    bm_t1 = os.path.join(run_dir, "brainmask_t1.nii.gz")
    wm_t1 = os.path.join(run_dir, "wm_t1.nii.gz")
    cs_t1 = os.path.join(run_dir, "csf_t1.nii.gz")
    _fs_mgz_to_nii(bm_mgz, bm_t1)
    _fs_mgz_to_nii(wm_mgz, wm_t1)
    _fs_mgz_to_nii(csf_mgz, cs_t1)

    # Transform T1 --> EPI space (NN):
    itk_txt = os.path.join(run_dir, "t1toepi_itk.txt")
    _regdat_to_itk_t1_to_epi(regdat, epi, t1_brain, itk_txt, run_dir)

    brain_epi = os.path.join(run_dir, "brainmask_epi.nii.gz")
    wm_epi    = os.path.join(run_dir, "wm_epi.nii.gz")
    csf_epi   = os.path.join(run_dir, "csf_epi.nii.gz")
    _ants_apply_nn(bm_t1, epi, itk_txt, brain_epi)
    _ants_apply_nn(wm_t1, epi, itk_txt, wm_epi)
    _ants_apply_nn(cs_t1, epi, itk_txt, csf_epi)

    epi_img = nib.load(epi)
    bm_img  = nib.load(brain_epi)
    wm_img  = nib.load(wm_epi)
    cs_img  = nib.load(csf_epi)

    # (OPTIONAL) Clip T1-derived brainmask to EPI support (to tighten edges):
    if CLIP_TO_EPI_SUPPORT:
        supp = _epi_support_mask(epi_img).get_fdata() > 0
        b = (bm_img.get_fdata() > 0.5) & supp
        w = (wm_img.get_fdata() > 0.5) & b
        c = (cs_img.get_fdata() > 0.5) & b
        bm_img = nib.Nifti1Image(b.astype(np.uint8), epi_img.affine, epi_img.header)
        wm_img = nib.Nifti1Image(w.astype(np.uint8), epi_img.affine, epi_img.header)
        cs_img = nib.Nifti1Image(c.astype(np.uint8), epi_img.affine, epi_img.header)

    # Optional erosions (currently '0' by default):
    wm_img = _maybe_erode(wm_img,  ERODE_WM_VOX)
    cs_img = _maybe_erode(cs_img,  ERODE_CSF_VOX)

    # GM = brain ∧ ¬WM ∧ ¬CSF:
    b = bm_img.get_fdata() > 0.5
    w = wm_img.get_fdata() > 0.5
    c = cs_img.get_fdata() > 0.5
    g = b & (~w) & (~c)
    gm_img = nib.Nifti1Image(g.astype(np.uint8), epi_img.affine, epi_img.header)
    gm_epi = os.path.join(run_dir, "gm_epi.nii.gz")
    nib.save(gm_img, gm_epi)

    # Compute fractions on EPI lattice:
    tot = int(b.sum()); wm_cnt = int(w.sum()); cs_cnt = int(c.sum())
    wm_p = 100 * wm_cnt / max(tot, 1)
    csf_p = 100 * cs_cnt / max(tot, 1)
    gm_p = 100 - wm_p - csf_p
    print(f"[{tag}]  WM={wm_p:.1f}%  GM={gm_p:.1f}%  CSF={csf_p:.1f}%  | counts (brain,WM,CSF)=({tot:,},{wm_cnt:,},{cs_cnt:,})")

    # Generate QC overlays:
    png = BBR_QC_DIR / f"{tag}_06_fs_masks_qc.png"
    png = str(png)
    disp = plotting.plot_anat(
        epi_img,
        title=f"{tag} — FS→EPI (ANTs ITK)  WM={wm_p:.1f}%  GM={gm_p:.1f}%  CSF={csf_p:.1f}%",
        display_mode='ortho', annotate=False, draw_cross=False, dim=False)
    disp.add_contours(bm_img, levels=[0.5], linewidths=1.0, colors='lime')
    disp.add_contours(wm_img, levels=[0.5], linewidths=1.0, colors='blue')
    disp.add_contours(cs_img, levels=[0.5], linewidths=1.0, colors='gold')
    disp.savefig(png, dpi=150); disp.close()
    print(f"[BBR QC] {tag}: wrote {png}")

    summ.append({
        "subject_ID": subj, "variant": f"ants-{'cli' if ANTS_APPLY else 'py'}",
        "wm_%brain": round(wm_p, 1), "gm_%brain": round(gm_p, 1), "csf_%brain": round(csf_p, 1),
        "brainmask_epi": brain_epi, "wm_epi": wm_epi, "csf_epi": csf_epi, "gm_epi": gm_epi,
        "qc_png": png})

summary_df = pd.DataFrame(summ).sort_values("subject_ID").reset_index(drop=True)
with pd.option_context("display.max_colwidth", 140):
    display(summary_df[["subject_ID","variant","wm_%brain","gm_%brain","csf_%brain","qc_png"]])

---------
Next step:
- builds Friston-24 regressors from motion
- computes aCompCor from WM+CSF in EPI space
- censors high-motion / high-DVARS frames (via sample_mask)
- applies band-pass (or high-pass only) + detrend/standardize in one go (using nilearn.signal.clean)
- saves the cleaned 4D timeseries and all confound files
- writes simple QC plots (motion/FD/DVARS with censored frames marked)
- optionally computes and saves a tSNR map (and an optional tSNR mask; off by default)

**NOTE:** The SCRUB_MODE toggle parameter controls how we handle "censored" frames:
- **"drop**"** removes these frames, thereby changing the total length of the recording in the process -- this is not appropriate for time-series analyses, since it changes the meaning of the time points across files!
- **"spike"** *"adds one extra nuisance regressor per bad frame to the design matrix" [...] "Each of those regressors is a vector of all zeros except a single 1 at the bad frame’s timepoint; during regression, the model is free to soak up (explain away) any weird signal at those bad frames via those spike columns.*
- **"interp"** performs interpolation for "censored" frames, which can be fine when only 1-3 consecutive frames are dropped, but which can become inaccurate for longer "runs" of dropped frames.

In [ ]:
# === Denoising on EPI grid (Friston-24 + aCompCor + scrubbing) with clear QC
# Panels: Motion(6), FD(+red), DVARS(+orange), Global signal (z)


# Tidy up (i.e. disable) printed warnings from older nilearn/deps:
warnings.filterwarnings("ignore", category=FutureWarning,     message=".*zscore.*will be replaced.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*default strategy for standardize.*")
warnings.filterwarnings("ignore", category=FutureWarning,     message="In the future `np.bool` will be defined.*")
warnings.filterwarnings("ignore", category=RuntimeWarning,    message=".*invalid value encountered in divide.*")

# Tiny compatibility shim for older ITK expecting np.bool/np.int/np.float:
if not hasattr(np, "bool"):   np.bool   = bool    # type: ignore[attr-defined]
if not hasattr(np, "int"):    np.int    = int     # type: ignore[attr-defined]
if not hasattr(np, "float"):  np.float  = float   # type: ignore[attr-defined]


# Pre-check for required globals:
for var in ("fMRI_runs", "BASE_DIRECTORY", "BBR_OUTPUT_DIR", "DENOISING_QC_DIR"):
    if var not in globals():
        raise RuntimeError(f"Missing global '{var}'. Run earlier cells first.")

# Set filepaths:
def _bold_path(tag):
    return os.path.join(str(SDC_DIRECTORY), tag, "bold_mc_sdc.nii.gz")

def _confounds_path(tag):
    return os.path.join(BASE_DIRECTORY, "confounds", f"{tag}_confounds.tsv")

def _xforms_dir(tag):
    return os.path.join(BASE_DIRECTORY, "motion_xforms", tag)

def _bbr_mask_paths(tag):
    rd = os.path.join(str(BBR_OUTPUT_DIR), tag)
    return (os.path.join(rd, "brainmask_epi.nii.gz"),
            os.path.join(rd, "wm_epi.nii.gz"),
            os.path.join(rd, "csf_epi.nii.gz"))

# Other helper functions:
def _zscore_sample(X, axis=0):
    """Sample z-score (ddof=1), NaN-safe. Zero-var -> 0."""
    X = np.asarray(X, dtype=np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        mu = np.nanmean(X, axis=axis, keepdims=True)
        sd = np.nanstd(X,  axis=axis, ddof=1, keepdims=True)
        bad = ~np.isfinite(sd) | (sd == 0)
        sd[bad] = 1.0
        Z = (X - mu) / sd
        Z[~np.isfinite(Z)] = 0.0
    return Z

def _euler_from_R(R):
    sy = np.sqrt(R[0,0]*R[0,0] + R[1,0]*R[1,0])
    singular = sy < 1e-6
    if not singular:
        rx = np.arctan2(R[2,1], R[2,2]); ry = np.arctan2(-R[2,0], sy); rz = np.arctan2(R[1,0], R[0,0])
    else:
        rx = np.arctan2(-R[1,2], R[1,1]); ry = np.arctan2(-R[2,0], sy); rz = 0.0
    return rx, ry, rz

def _motion6_from_itk_xforms(xdir, n_drop):
    try:
        import itk
    except Exception as e:
        raise RuntimeError(f"Failed to import ITK: {e}")
    files = sorted(glob.glob(os.path.join(xdir, "mc_*_0GenericAffine.mat")))
    if not files:
        raise FileNotFoundError(f"No transforms found in {xdir}")
    dx= []; dy= []; dz= []; rx= []; ry= []; rz= []
    for fp in files:
        tx_list = itk.transformread(fp)
        if not tx_list:
            raise RuntimeError(f"Cannot read transform from {fp}")
        tx = tx_list[0]
        M = np.array(itk.array_from_matrix(tx.GetMatrix()), dtype=np.float64).reshape(3,3)
        t = np.array(list(tx.GetTranslation()), dtype=np.float64)
        c = np.array(list(tx.GetCenter()), dtype=np.float64) if hasattr(tx, "GetCenter") else np.zeros(3, dtype=np.float64)
        t_eff = t + (np.eye(3) - M) @ c
        rxs, rys, rzs = _euler_from_R(M)
        dx.append(t_eff[0]); dy.append(t_eff[1]); dz.append(t_eff[2])
        rx.append(rxs);      ry.append(rys);      rz.append(rzs)
    dx = np.asarray(dx); dy = np.asarray(dy); dz = np.asarray(dz)
    rx = np.asarray(rx); ry = np.asarray(ry); rz = np.asarray(rz)
    if n_drop > 0:
        dx = dx[n_drop:]; dy = dy[n_drop:]; dz = dz[n_drop:]
        rx = rx[n_drop:]; ry = ry[n_drop:]; rz = rz[n_drop:]
    return np.vstack([dx,dy,dz,rx,ry,rz]).T

def _friston24(m6):
    dm6 = np.vstack([np.zeros((1,6)), np.diff(m6, axis=0)])
    return np.hstack([m6, dm6, m6**2, dm6**2])

def _acompcor_components(data_4d, mask_img, n_comp):
    if n_comp <= 0: return np.empty((data_4d.shape[3], 0), dtype=np.float32)
    mask = mask_img.get_fdata() > 0.5
    if mask.sum() == 0: return np.empty((data_4d.shape[3], 0), dtype=np.float32)
    T = data_4d.shape[3]
    vox = data_4d[mask].reshape(-1, T).T  # (T x N)
    col_mu = np.nanmean(vox, axis=0, keepdims=True)
    bad    = ~np.isfinite(vox)
    if bad.any():
        idx_cols = np.where(bad)[1]
        vox[bad] = np.take(col_mu.squeeze(), idx_cols)
    if np.nanvar(vox, axis=0).sum() == 0:
        return np.empty((T, 0), dtype=np.float32)
    vox = _zscore_sample(vox, axis=0)
    if np.nanvar(vox, axis=0).sum() == 0:
        return np.empty((T, 0), dtype=np.float32)
    pca = PCA(n_components=min(n_comp, vox.shape[1]), svd_solver="randomized", random_state=RANDOM_SEED)
    return pca.fit_transform(vox).astype(np.float32)

def _expand_bad(bad_bool, n):
    """Expand True indices by n neighbors on each side (1D)."""
    if n <= 0: return bad_bool
    w = np.ones(2*n + 1, dtype=int)
    conv = np.convolve(bad_bool.astype(int), w, mode="same")
    return conv > 0

def _make_sample_mask(fd, dvars, fd_thr, dvars_thr, fd_abs=None, dvars_abs=None):
    keep = np.ones_like(fd, dtype=bool)
    if np.isfinite(fd_thr):    keep &= (fd    <= fd_thr)
    if fd_abs is not None:     keep &= (fd    <= float(fd_abs))
    if np.isfinite(dvars_thr): keep &= (dvars <= dvars_thr)
    if dvars_abs is not None:  keep &= (dvars <= float(dvars_abs))
    if keep.size > 0: keep[0] = True  # always keep first frame
    return keep

def _max_run_length(b):
    """Longest consecutive run of True in 1D boolean array."""
    if b.size == 0: return 0
    bi = b.astype(int)
    changes = np.diff(np.concatenate(([0], bi, [0])))
    starts = np.where(changes == 1)[0]
    ends   = np.where(changes == -1)[0]
    lengths = (ends - starts)
    return int(lengths.max()) if lengths.size else 0

def _linear_interp_fill(X, keep_mask):
    """
    Fills bad rows (time) in 2D array X (T x N) by linear interpolation along time, per column.
    Edges use nearest hold (left/right).
    """
    T, N = X.shape
    t = np.arange(T)
    keep_idx = np.where(keep_mask)[0]
    if keep_idx.size < 2:
        out = X.copy()
        if keep_idx.size == 1:
            out[:] = X[keep_idx[0]]
        else:
            out[:] = 0.0
        return out
    left_vals  = X[keep_idx[0], :]
    right_vals = X[keep_idx[-1], :]
    out = np.empty_like(X)
    for j in range(N):
        fp = X[keep_idx, j]
        out[:, j] = np.interp(t, keep_idx, fp, left=left_vals[j], right=right_vals[j])
    return out


# ________________________________________________________________________________________________________
# Main execution:

rows = fMRI_runs.to_dict(orient="records")
summ_rows = []

for r in rows:
    subj, grp, sess = str(r["subject_ID"]), str(r["group_ID"]), str(r["session_ID"])
    tag = f"{subj}_{sess}"

    bold_path = _bold_path(tag)
    conf_tsv  = _confounds_path(tag)
    xdir      = _xforms_dir(tag)
    brain_p, wm_p, csf_p = _bbr_mask_paths(tag)

    out_dir = Path(DENOISING_OUTPUT_DIR) / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    qc_png   = Path(DENOISING_QC_DIR) / f"{tag}_07_denoise_qc.png"
    bold_out = out_dir / "bold_denoised.nii.gz"

    # Check for required inputs:
    if not os.path.exists(bold_path):
        print(f"[WARN] {tag}: missing BOLD 4D at {bold_path} ; skipping."); continue
    if not os.path.exists(conf_tsv):
        print(f"[WARN] {tag}: missing confounds TSV at {conf_tsv} ; skipping."); continue
    if not (os.path.exists(brain_p) and os.path.exists(wm_p) and os.path.exists(csf_p)):
        print(f"[WARN] {tag}: missing EPI masks in {os.path.dirname(brain_p)} ; run BBR-3 first. Skipping.")
        continue

    # Load BOLD:
    img   = nib.load(bold_path)
    data  = img.get_fdata(dtype=np.float32)
    # ---- NEW: scrub NaNs/Infs immediately after load
    data  = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    if data.ndim != 4:
        print(f"[WARN] {tag}: BOLD not 4D ; skipping."); continue
    NX, NY, NZ, NT = data.shape  # <-- uses distinct names to avoid collisions

    # Load confounds (FD, DVARS, dummy drop count, TR):
    dfc = pd.read_csv(conf_tsv, sep="\t")
    if "n_dummy_dropped" not in dfc.columns or "tr_s" not in dfc.columns:
        raise RuntimeError(f"{tag}: confounds TSV missing required columns.")
    n_dummy = int(dfc["n_dummy_dropped"].iloc[0])
    TR      = float(dfc["tr_s"].iloc[0])

    # Trim dummies in BOLD (confounds already dropped them):
    if n_dummy > 0:
        data = data[..., n_dummy:]
        NT = data.shape[3]
        print(f"[{tag}] Trimmed first {n_dummy} dummy volumes. New T={NT}.")

    # FD & DVARS (length T_conf):
    fd    = dfc["framewise_displacement"].to_numpy(dtype=np.float32)
    dvars = dfc["dvars"].to_numpy(dtype=np.float32)

    # Motion 6 from ITK xforms (NB: already honors 'n_dummy'):
    try:
        m6 = _motion6_from_itk_xforms(xdir, n_dummy)
    except Exception as e:
        raise RuntimeError(f"{tag}: failed to read motion xforms from {xdir}: {e}")

    # Synchronize lengths robustly:
    T_conf = int(fd.shape[0])
    T_m6   = int(m6.shape[0])
    T_sync = min(NT, T_conf, T_m6)
    if not (NT == T_conf == T_m6):
        print(f"[{tag}] Length sync: BOLD T={NT}, confounds={T_conf}, motion6={T_m6} -> using T={T_sync}.")
    if NT != T_sync:       data = data[..., :T_sync]; NT = T_sync
    if T_conf != T_sync:   fd, dvars = fd[:T_sync], dvars[:T_sync]
    if T_m6   != T_sync:   m6 = m6[:T_sync, :]

    # Set thresholds:
    fd_thr    = np.nanpercentile(fd,    FD_RELAXED_PERC) if np.isfinite(fd).any()    else np.nan
    dvars_thr = np.nanpercentile(dvars, DVARS_RELAXED_P) if np.isfinite(dvars).any() else np.nan

    # Base bad masks:
    bad_fd    = np.isfinite(fd)    & (fd    > fd_thr)
    bad_dvars = np.isfinite(dvars) & (dvars > dvars_thr)
    if FD_ABS_MM is not None:   bad_fd    |= (fd    > float(FD_ABS_MM))
    if DVARS_ABS is not None:   bad_dvars |= (dvars > float(DVARS_ABS))
    bad_any = bad_fd | bad_dvars

    # Neighbor expansion:
    bad_any_exp = _expand_bad(bad_any, NEIGHBOR_EXPAND)
    keep_mask   = ~bad_any_exp
    keep_mask   = np.asarray(keep_mask, dtype=bool)
    if keep_mask.size > 0: keep_mask[0] = True

    # Max consecutive flagged runs (for accounting):
    max_run_flagged = _max_run_length(bad_any_exp)

    # Prepare masks and signals:
    brain_img = nib.load(brain_p)
    wm_img    = nib.load(wm_p)
    csf_img   = nib.load(csf_p)

    brain_mask = brain_img.get_fdata() > 0.5
    wm_mask    = wm_img.get_fdata()   > 0.5
    csf_mask   = csf_img.get_fdata()  > 0.5
    if brain_mask.sum() == 0:
        raise RuntimeError(f"{tag}: brain mask has 0 voxels.")
    gm_mask = brain_mask & (~wm_mask) & (~csf_mask)

    data_2d  = data.reshape(-1, NT).T               # <-- (T x V)
    flat_b   = brain_mask.reshape(-1)
    brain_idx_all = np.where(flat_b)[0]
    brain_ts_all  = data_2d[:, brain_idx_all]       # <-- (T x B)

    # Drop non-finite brain voxels from regression (but remember them):
    finite_cols = np.isfinite(brain_ts_all).all(axis=0)
    dropped_vox = int((~finite_cols).sum())
    brain_idx   = brain_idx_all[finite_cols]
    brain_ts    = brain_ts_all[:, finite_cols]

    # Compute global (GM or brain) signal:
    if GSR_TISSUE.lower() == "brain":
        gs_mask_idx = np.where(brain_mask.reshape(-1))[0]
    else:
        gs_mask_idx = np.where(gm_mask.reshape(-1))[0]
    if gs_mask_idx.size:
        gs_raw = np.nanmean(data_2d[:, gs_mask_idx], axis=1)
    else:
        gs_raw = np.zeros(NT, dtype=np.float32)
    gs_z = _zscore_sample(gs_raw[:, None]).ravel()

    # Perform aCompCor on EPI grid:
    wm_cc  = _acompcor_components(data, wm_img,  ACOMPCOR_N_WM)
    csf_cc = _acompcor_components(data, csf_img, ACOMPCOR_N_CSF)

    # Friston-24 from motion:
    R24 = _friston24(m6)

    # Build confounds (always NT rows):
    conf_list = [R24, wm_cc, csf_cc]
    if APPLY_GSR:
        conf_list.append(gs_z[:, None].astype(np.float32))
    if any(c.size > 0 for c in conf_list):
        X_used = np.hstack([c for c in conf_list if c.size > 0]).astype(np.float32)
        X_used = _zscore_sample(X_used, axis=0)
        X_used = np.nan_to_num(X_used, nan=0.0, posinf=0.0, neginf=0.0)
    else:
        X_used = np.zeros((NT, 0), dtype=np.float32)

    # Spike regressors (length-preserving) if needed:
    spike_cols = 0
    if SCRUB_MODE.lower() == "spike":
        bad_idx = np.where(~keep_mask)[0]
        if bad_idx.size:
            Z = np.zeros((NT, bad_idx.size), dtype=np.float32)
            Z[bad_idx, np.arange(bad_idx.size)] = 1.0
            X_used = np.hstack([X_used, Z]) if X_used.size else Z
            spike_cols = int(bad_idx.size)

    # Standardize brain_ts columns (sample z):
    brain_ts = _zscore_sample(brain_ts, axis=0)

    # Denoising (per mode):
    if SCRUB_MODE.lower() == "drop":
        cleaned_ts = clean(
            brain_ts,
            confounds=X_used,
            detrend=DETREND,
            standardize=False,
            standardize_confounds=False,
            low_pass=LP_HZ,
            high_pass=HP_HZ,
            t_r=TR,
            ensure_finite=True,
            filter="butterworth",
            sample_mask=keep_mask,
            extrapolate=False)  # <-- avoids cubic-spline extrapolation
        kept_count = int(keep_mask.sum())
        cleaned_2d = data_2d[keep_mask, :].copy()
        cleaned_2d[:, brain_idx] = cleaned_ts
        if dropped_vox > 0:
            bad_idx_vox = brain_idx_all[~finite_cols]
            cleaned_2d[:, bad_idx_vox] = 0.0

    elif SCRUB_MODE.lower() == "interp":
        interp_ts = _linear_interp_fill(brain_ts, keep_mask)
        cleaned_ts = clean(
            interp_ts,
            confounds=X_used,
            detrend=DETREND,
            standardize=False,
            standardize_confounds=False,
            low_pass=LP_HZ,
            high_pass=HP_HZ,
            t_r=TR,
            ensure_finite=True,
            filter="butterworth",
            extrapolate=False)  # <-- no sample_mask, but be explicit
        kept_count = int(keep_mask.sum())
        cleaned_2d = data_2d.copy()
        cleaned_2d[:, brain_idx] = cleaned_ts
        if dropped_vox > 0:
            bad_idx_vox = brain_idx_all[~finite_cols]
            cleaned_2d[:, bad_idx_vox] = 0.0

    else:  # 'spike' (default):
        cleaned_ts = clean(
            brain_ts,
            confounds=X_used,
            detrend=DETREND,
            standardize=False,
            standardize_confounds=False,
            low_pass=LP_HZ,
            high_pass=HP_HZ,
            t_r=TR,
            ensure_finite=True,
            filter="butterworth",
            extrapolate=False)
        kept_count = int(keep_mask.sum())
        cleaned_2d = data_2d.copy()
        cleaned_2d[:, brain_idx] = cleaned_ts
        if dropped_vox > 0:
            bad_idx_vox = brain_idx_all[~finite_cols]
            cleaned_2d[:, bad_idx_vox] = 0.0

    # Derive T_out from the array we are about to re-shape:
    T_out = int(cleaned_2d.shape[0])
    # Save 4D using original spatial lattice:
    NXi, NYi, NZi, Ti = int(NX), int(NY), int(NZ), int(T_out)
    cleaned_4d = cleaned_2d.T.reshape(NXi, NYi, NZi, Ti).astype(np.float32)
    
    # scrub NaNs/Infs right before save:
    cleaned_4d = np.nan_to_num(cleaned_4d, nan=0.0, posinf=0.0, neginf=0.0)
    nib.save(nib.Nifti1Image(cleaned_4d, img.affine, img.header), str(bold_out))

    # Create QC figures:
    t = np.arange(NT)
    dx,dy,dz,rx,ry,rz = m6.T
    rx_deg, ry_deg, rz_deg = np.rad2deg(rx), np.rad2deg(ry), np.rad2deg(rz)

    fig = plt.figure(figsize=(14, 9))
    gs  = fig.add_gridspec(4, 1, height_ratios=[2.0, 1.6, 1.6, 1.3], hspace=0.35)

    ax1 = fig.add_subplot(gs[0,0])
    ax1.plot(t, dx, label='dx (mm)')
    ax1.plot(t, dy, label='dy (mm)')
    ax1.plot(t, dz, label='dz (mm)')
    ax1.plot(t, rx_deg, label='rx (deg)', alpha=0.85)
    ax1.plot(t, ry_deg, label='ry (deg)', alpha=0.85)
    ax1.plot(t, rz_deg, label='rz (deg)', alpha=0.85)
    ax1.set_ylabel('Rigid motion')
    ax1.set_title(f"{tag} - Denoising QC [{SCRUB_MODE}]")
    ax1.legend(ncol=3, fontsize=9)

    ax2 = fig.add_subplot(gs[1,0], sharex=ax1)
    ax2.plot(t, fd, label='FD (mm)')
    if np.isfinite(fd_thr): ax2.axhline(fd_thr, color='C0', linestyle='--', linewidth=1, label=f'FD thr p{FD_RELAXED_PERC}')
    if FD_ABS_MM is not None: ax2.axhline(FD_ABS_MM, color='C0', linestyle=':', linewidth=1, label=f'FD abs {FD_ABS_MM}mm')
    fd_idx = np.where((np.isfinite(fd)) & (~keep_mask) & (fd > 0))[0]
    if fd_idx.size:
        ax2.scatter(fd_idx, fd[fd_idx], s=18, color='red', marker='o', label='FD-flagged')
    ax2.set_ylabel('FD (mm)')
    ax2.legend(fontsize=9, loc='upper right')

    ax3 = fig.add_subplot(gs[2,0], sharex=ax1)
    ax3.plot(t, dvars, color='C1', label='DVARS')
    if np.isfinite(dvars_thr): ax3.axhline(dvars_thr, color='C1', linestyle='--', linewidth=1, label=f'DVARS thr p{DVARS_RELAXED_P}')
    if DVARS_ABS is not None: ax3.axhline(DVARS_ABS, color='C1', linestyle=':', linewidth=1, label=f'DVARS abs {DVARS_ABS}')
    dv_idx = np.where((np.isfinite(dvars)) & (~keep_mask) & (dvars > 0))[0]
    if dv_idx.size:
        ax3.scatter(dv_idx, dvars[dv_idx], s=18, color='orange', marker='o', label='DVARS-flagged')
    ax3.set_ylabel('DVARS')
    ax3.legend(fontsize=9, loc='upper right')

    ax4 = fig.add_subplot(gs[3,0], sharex=ax1)
    ax4.plot(t, _zscore_sample(gs_z[:,None]).ravel(), label=f'Global signal (z), {GSR_TISSUE}')
    for i in np.where(~keep_mask)[0]:
        ax4.axvspan(i-0.5, i+0.5, color='k', alpha=0.05)
    ax4.set_ylabel('GS (z)'); ax4.set_xlabel('Frame'); ax4.legend(fontsize=9, loc='upper right')

    wm_used = wm_cc.shape[1]; csf_used = csf_cc.shape[1]
    conf_cols = X_used.shape[1]
    dropped_frames = int((~keep_mask).sum())
    kept_count_pct = round(100 * kept_count / max(NT, 1), 1)

    # DOF accounting:
    frames_used = int(keep_mask.sum()) if SCRUB_MODE.lower() == "drop" else int(NT)
    regressors_used = int(conf_cols)
    remaining_dof = int(max(frames_used - regressors_used, 0))

    Path(DENOISING_QC_DIR).mkdir(parents=True, exist_ok=True)
    fig.suptitle(
        f"mode={SCRUB_MODE} | kept {kept_count}/{NT} frames ({kept_count_pct}%) | output T={T_out} | "
        f"confounds: {conf_cols} (R24 + WM:{wm_used} + CSF:{csf_used}"
        f"{' + GSR' if APPLY_GSR else ''}{' + spikes:'+str(spike_cols) if spike_cols else ''}) | "
        f"DOF approx: {remaining_dof} | flagged: {dropped_frames} (max run {max_run_flagged})",
        y=0.995, fontsize=10.5)
    fig.savefig(str(qc_png), dpi=150, bbox_inches='tight'); plt.close(fig)

    print(
        f"[DENOISE] {tag}: saved {bold_out}\n"
        f"          kept {kept_count}/{NT} frames | output T={T_out} | confounds: {conf_cols} cols "
        f"(R24 + WM:{wm_used} + CSF:{csf_used}{' + GSR' if APPLY_GSR else ''}{' + spikes:'+str(spike_cols) if spike_cols else ''}) "
        f"| dropped non-finite voxels: {dropped_vox}\n"
        f"          flagged frames: {dropped_frames} (max consecutive {max_run_flagged})\n"
        f"          approx DOF remaining: {remaining_dof}\n"
        f"          QC: {qc_png}")

    # Construct summary row:
    summ_rows.append({
        "subject_ID": subj, "TR": TR,
        "mode": SCRUB_MODE, "T_in": int(NT), "T_out": int(T_out),
        "kept": int(kept_count), "dropped": dropped_frames, "max_run_dropped": max_run_flagged,
        "kept_pct": kept_count_pct,
        "conf_cols": int(conf_cols), "spike_cols": int(spike_cols),
        "dof_frames_used": int(frames_used), "dof_regressors": int(regressors_used), "dof_remaining": int(remaining_dof),
        "bold_out": str(bold_out), "qc_png": str(qc_png),
        "WM_cc": int(wm_used), "CSF_cc": int(csf_used), "GSR": bool(APPLY_GSR), "neighbor_expand": int(NEIGHBOR_EXPAND)})

summary_df = pd.DataFrame(summ_rows).sort_values(["subject_ID"]).reset_index(drop=True)
with pd.option_context("display.max_colwidth", 140):
    display(summary_df)

# Save output dataframe:
summary_output_path = Path(DENOISING_QC_DIR) / "_denoising_summary.csv"
summary_df.to_csv(summary_output_path, index=False)

----------